# MANUAL RNN

In [ ]:
import torch 
import torch.nn as nn 
import torch.nn.functional as F
import torch.optim as optim
from torchinfo import summary

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split


In [ ]:
class ManualRNN(nn.Module) :
    def __init__(self, vocab_size, hidden_size=3, output_size = 1) :
        super().__init__()

        self.vocab_size = vocab_size

        self.hidden_size = hidden_size
        self.output_size = output_size


        self.W_xh = nn.Parameter(torch.randn(size=(self.vocab_size, self.hidden_size)) * 0.1)
        self.W_hh = nn.Parameter(torch.randn(size=(self.hidden_size, self.hidden_size)) * 0.1)
        self.W_hy = nn.Parameter(torch.randn(size=(self.hidden_size, self.output_size)) * 0.1)

        self.b_hl = nn.Parameter(torch.zeros(size=(self.hidden_size,)) )
        self.b_yl = nn.Parameter(torch.zeros(size=(self.output_size,)))

    def forward(self, x, h_prev = None) :
        # shape of x = (batch,timesteps,features)
        
        batch_size,timesteps_count,features_size = x.shape

        if h_prev == None : 
            h_prev = torch.zeros(size=(batch_size, self.hidden_size),
                                    device = x.device,
                                    dtype=x.dtype)
            
        outputs = []
        h_states = []

        for timestep in range(timesteps_count) : 

            input = x[ :, timestep, : ]
             
            h_t = torch.tanh((input @ self.W_xh) + (h_prev @ self.W_hh) + self.b_hl)
            y_t = torch.sigmoid((h_t @ self.W_hy) + self.b_yl)

            h_prev = h_t
            outputs.append(y_t)
            h_states.append(h_t)

        return outputs, h_states


In [68]:
model = ManualRNN(vocab_size=10, hidden_size=5, output_size=1)

for parameter_name, parameter in model.named_parameters():
    print(f"{parameter_name:5s} -> {tuple(parameter.shape)}")


W_xh  -> (10, 5)
W_hh  -> (5, 5)
W_hy  -> (5, 1)
b_hl  -> (5,)
b_yl  -> (1,)
